# SSUSA–IUCN Species Comparison with Trait & Common-Name Enrichment

Clean Snapshot USA camera-trap data, spatially intersect with IUCN range polygons (1 km point buffers), then enrich each species with PanTHERIA life-history traits and ITIS standardized common names.

**Pipeline:**
1. Setup & configuration
2. Load raw SSUSA sequences & deployments
3. Clean taxonomy & build scientific names
4. Merge sequences with deployments
5. Standardize & filter species (Mammalia only; remove humans, domestics, non-native)
6. Remap species names to IUCN taxonomy
7. IUCN spatial join — 1 km point buffers per deployment
8. Merge PanTHERIA life-history traits
9. Merge ITIS common names
10. Export species-per-array table → `data/traits/species_per_array_iucn_traits.csv`
11. Export unique species list → `data/traits/unique_species_traits.csv`
12. Diagnostics — identify species missing traits or common names


**Data sources:**
- **SSUSA** — Snapshot USA camera-trap sequences & deployments
- **IUCN Red List** — [Spatial data](https://www.iucnredlist.org/resources/spatial-data-download): terrestrial mammal range polygons
- **PanTHERIA** — [Jones et al. 2009](https://esapubs.org/archive/ecol/E090/184/): species-level ecology & life-history traits (MSW05 taxonomy)
- **ITIS** — [Integrated Taxonomic Information System](https://www.itis.gov/): standardized English common names

## 1. Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import re
import random
import sqlite3

In [ ]:
# Input / output directories
data_dir = '../data/ssusa'          # raw SSUSA data lives here
output_dir = '../data/traits'       # all outputs go here
os.makedirs(output_dir, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


random.seed(42)
np.random.seed(42)

## 2. Load Raw SSUSA Data

In [3]:
sequences_path = os.path.join(data_dir, 'ssusa_finalsequences.csv')
deployments_path = os.path.join(data_dir, 'ssusa_finaldeployments.csv')

# Load sequences
sequences_df = pd.read_csv(
    sequences_path,
    na_values=['NA', ' '],
    dtype={'Year': 'Int64', 'Sequence_ID': 'object', 'Group_Size': 'Int64'},
    keep_default_na=True,
    parse_dates=['Start_Time', 'End_Time']
)
print(f'Sequences loaded: {len(sequences_df):,} records')

# Load deployments
deployments_df = pd.read_csv(
    deployments_path,
    na_values=['NA', ' '],
    dtype={'Year': 'Int64', 'Survey_Nights': 'float64', 'Latitude': 'float64', 'Longitude': 'float64'},
    keep_default_na=True,
    parse_dates=['Start_Date', 'End_Date']
)
print(f'Deployments loaded: {len(deployments_df):,} records')

Sequences loaded: 987,979 records
Deployments loaded: 9,679 records


## 3. Clean Taxonomy & Build Scientific Names

In [4]:
# Drop rows missing required taxonomy columns
required_cols = ['Class', 'Order', 'Family', 'Genus', 'Species', 'Common_Name']
sequences_df[required_cols] = sequences_df[required_cols].replace(' ', pd.NA)

initial_count = len(sequences_df)
sequences_df = sequences_df.dropna(subset=required_cols)
print(f'Dropped {initial_count - len(sequences_df):,} rows with missing taxonomy → {len(sequences_df):,} remaining')

Dropped 96,032 rows with missing taxonomy → 891,947 remaining


In [5]:
# Create standardized scientific name
sequences_df['Sci_Name'] = (
    sequences_df['Genus'].str.capitalize().str.strip() + ' ' +
    sequences_df['Species'].str.lower().str.strip()
).str.strip()

# Move Sci_Name to after Common_Name
sci_name_col = sequences_df.pop('Sci_Name')
common_name_index = sequences_df.columns.get_loc('Common_Name')
sequences_df.insert(common_name_index + 1, 'Sci_Name', sci_name_col)
print(f'Sci_Name created. Sample: {sequences_df["Sci_Name"].iloc[:3].tolist()}')

Sci_Name created. Sample: ['Ursus arctos', 'Ursus arctos', 'Ursus arctos']


## 4. Merge Sequences & Deployments

In [6]:
common_cols = list(set(sequences_df.columns).intersection(deployments_df.columns))
print(f'Merging on columns: {common_cols}')

merged_df = pd.merge(sequences_df, deployments_df, on=common_cols, how='inner')
merged_df = merged_df.drop_duplicates()
print(f'Merged dataset: {len(merged_df):,} records')

Merging on columns: ['Project', 'Camera_Trap_Array', 'Deployment_ID', 'Year']
Merged dataset: 885,087 records


## 5. Standardize & Filter Species

In [7]:
# Standardize text columns to lowercase
prop_case_cols = [
    'Class', 'Order', 'Family', 'Genus', 'Species', 'Habitat',
    'Development_Level', 'Feature_Type', 'Common_Name', 'Age', 'Sex'
]

merged_df[['Age', 'Sex', 'Group_Size']] = merged_df[['Age', 'Sex', 'Group_Size']].replace(r'^\s*$', pd.NA, regex=True)
merged_df['Group_Size'] = pd.to_numeric(merged_df['Group_Size'], errors='coerce').fillna(0).astype(int)
merged_df['Age'] = merged_df['Age'].fillna('Unknown')
merged_df['Sex'] = merged_df['Sex'].fillna('Unknown')

for col in prop_case_cols:
    merged_df[col] = merged_df[col].str.lower()

print('Text columns standardized to lowercase.')

Text columns standardized to lowercase.


In [8]:
# Keep only Mammalia
merged_df = merged_df[merged_df['Class'] == 'mammalia']
print(f'After Mammalia filter: {len(merged_df):,}')

# Drop humans
merged_df = merged_df[
    (merged_df['Genus'].str.lower() != 'homo') &
    (merged_df['Species'].str.lower() != 'sapiens')
].reset_index(drop=True)
print(f'After dropping humans: {len(merged_df):,}')

After Mammalia filter: 857,003
After dropping humans: 772,478


In [9]:
# Drop domestic species
domestic_common = [
    'domestic dog', 'domestic cat', 'domestic cattle', 'domestic horse',
    'domestic sheep', 'domestic pig', 'domestic goat', 'domestic donkey',
    'brown rat', 'house rat', 'house mouse'
]
merged_df = merged_df[~merged_df['Common_Name'].str.lower().isin(domestic_common)].reset_index(drop=True)
print(f'After dropping domestics: {len(merged_df):,}')

After dropping domestics: 715,065


In [10]:
# Standardize common name for prairie dogs
merged_df.loc[merged_df['Sci_Name'] == 'Cynomys ludovicianus', 'Common_Name'] = 'black-tailed prairie dog'
print('Standardized prairie dog common name.')

Standardized prairie dog common name.


## 6. Remove Non-native Species & Remap to IUCN Taxonomy

In [11]:
# Drop non-native and/or data-poor species
species_to_drop = [
    'boselaphus tragocamelus', 'lama glama', 'lariscus insignis',
    'oryx gazella', 'urva javanica', 'sus scrofa'
]
merged_df = merged_df[~merged_df['Sci_Name'].str.lower().isin(species_to_drop)]
print(f'After dropping non-native/sparse species: {len(merged_df):,}')

After dropping non-native/sparse species: 698,887


In [12]:
# Rename species to match IUCN taxonomy
iucn_replace_map = {
    'mustela frenata': 'neogale frenata',
    'myodes gapperi': 'clethrionomys gapperi',
    'neovison vison': 'neogale vison',
    'pekania pennanti': 'martes pennanti'
}

merged_df['Sci_Name'] = merged_df['Sci_Name'].str.lower().replace(iucn_replace_map)

unique_species = merged_df['Sci_Name'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
print(f'Unique species after IUCN rename: {len(unique_species)}')

Unique species after IUCN rename: 113


## 7. IUCN Spatial Join (1 km Point Buffers)

For each camera-trap deployment, create a **1 km radius buffer** around its coordinates, then spatially intersect with IUCN terrestrial-mammal range polygons. Results are aggregated to the array level and outer-joined with SSUSA observations to produce a species × array table labelled:

| Category | Meaning |
|---|---|

| **both** | Detected by SSUSA cameras **and** inside IUCN predicted range || **iucn_only** | In IUCN range but **not** detected by cameras at that array |
| **ssusa_only** | Detected by cameras but **not** in IUCN range at that array |

In [ ]:
# Configuration
point_buffer_km = 1.0  # Buffer radius around each deployment point (km)
iucn_path = '../data/MAMMALS_TERRESTRIAL_ONLY/MAMMALS_TERRESTRIAL_ONLY.shp'

# Helper: normalize species names
def _normalize_species(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).strip()
    text = re.sub(r'\s+', ' ', text).lower()
    return text

In [14]:
# Load IUCN shapefile
iucn = gpd.read_file(iucn_path)
if iucn.crs is None:
    iucn = iucn.set_crs('EPSG:4326')
else:
    iucn = iucn.to_crs('EPSG:4326')

# Clean geometries
iucn = iucn[iucn.geometry.notnull()].copy()
if not iucn.is_valid.all():
    iucn['geometry'] = iucn.geometry.buffer(0)

print(f'IUCN loaded: {len(iucn):,} range polygons')

IUCN loaded: 12,703 range polygons


In [15]:
# Auto-detect IUCN species column
iucn_species_candidates = ['SCI_NAME', 'Sci_Name', 'binomial', 'scientific', 'species', 'Species']
iucn_species_col = None
lower_map = {c.lower(): c for c in iucn.columns}
for cand in iucn_species_candidates:
    if cand.lower() in lower_map:
        iucn_species_col = lower_map[cand.lower()]
        break
if iucn_species_col is None:
    raise ValueError('Could not detect IUCN species name column.')

# Normalize IUCN species names
iucn['Sci_Name'] = iucn[iucn_species_col].map(_normalize_species)
iucn = iucn.dropna(subset=['Sci_Name'])
iucn = iucn[['Sci_Name', 'geometry']]

print(f'IUCN species column: {iucn_species_col}')
print(f'Unique IUCN species: {iucn["Sci_Name"].nunique():,}')

IUCN species column: sci_name
Unique IUCN species: 5,683


In [16]:
# Create GeoDataFrame from deployment points
df_points = merged_df.dropna(subset=['Latitude', 'Longitude']).copy()
df_points['Latitude'] = pd.to_numeric(df_points['Latitude'], errors='coerce')
df_points['Longitude'] = pd.to_numeric(df_points['Longitude'], errors='coerce')
df_points = df_points.dropna(subset=['Latitude', 'Longitude'])

# Validate coordinate ranges
in_range = df_points['Latitude'].between(-90, 90) & df_points['Longitude'].between(-180, 180)
df_points = df_points[in_range].copy()

gdf_points = gpd.GeoDataFrame(
    df_points,
    geometry=gpd.points_from_xy(df_points['Longitude'], df_points['Latitude']),
    crs='EPSG:4326'
)

print(f'Valid deployment points: {len(gdf_points):,}')
print(f'Bounds (WGS84): {gdf_points.total_bounds}')

Valid deployment points: 698,887
Bounds (WGS84): [-152.37792      24.69486     -68.61159314   59.452635  ]


In [17]:
# Create 1 km point buffers around each individual deployment
array_col = 'Camera_Trap_Array'

gdf_points_m = gdf_points.to_crs(3857)
gdf_points_m['buffer'] = gdf_points_m.geometry.buffer(point_buffer_km * 1000.0)  # 1 km

# Build buffer GeoDataFrame with unique deployments (keep array info)
buffer_df = gdf_points_m[[array_col, 'Deployment_ID', 'buffer']].drop_duplicates(subset='Deployment_ID').copy()
gdf_buffers = gpd.GeoDataFrame(buffer_df, geometry='buffer', crs=3857)
gdf_buffers = gdf_buffers.rename_geometry('geometry')
gdf_buffers = gdf_buffers[gdf_buffers.geometry.notnull() & ~gdf_buffers.geometry.is_empty]

print(f'Point buffers created: {len(gdf_buffers):,} deployments ({point_buffer_km} km radius)')

Point buffers created: 9,255 deployments (1.0 km radius)


In [18]:
# Spatial join: intersect IUCN ranges with individual deployment point buffers
iucn_m = iucn.to_crs(3857)
iucn_m = iucn_m[iucn_m.geometry.notnull() & ~iucn_m.geometry.is_empty].copy()

iucn_in_buffers = gpd.sjoin(
    iucn_m,
    gdf_buffers,
    how='inner',
    predicate='intersects'
)

# Aggregate IUCN species to array level (union of all deployment buffers in an array)
iucn_array_species = (
    iucn_in_buffers[[array_col, 'Sci_Name']]
    .drop_duplicates()
    .rename(columns={array_col: 'Array', 'Sci_Name': 'species'})
)
iucn_array_species['in_iucn'] = True

print(f'IUCN-buffer intersections: {len(iucn_in_buffers):,}')
print(f'Unique IUCN species across all arrays: {iucn_array_species["species"].nunique()}')

IUCN-buffer intersections: 426,289
Unique IUCN species across all arrays: 304


In [19]:
# SSUSA observed species per array
snapshot_array_species = (
    df_points[[array_col, 'Sci_Name']]
    .dropna()
    .drop_duplicates()
    .rename(columns={array_col: 'Array', 'Sci_Name': 'species'})
)
snapshot_array_species['in_ssusa'] = True

print(f'Unique SSUSA species across all arrays: {snapshot_array_species["species"].nunique()}')

Unique SSUSA species across all arrays: 113


In [20]:
# Full outer join: SSUSA vs IUCN species per array
joint_df = snapshot_array_species.merge(
    iucn_array_species,
    on=['Array', 'species'],
    how='outer',
    sort=True,
)
joint_df['in_ssusa'] = joint_df['in_ssusa'].fillna(False)
joint_df['in_iucn'] = joint_df['in_iucn'].fillna(False)

# Label category
joint_df['Category'] = np.select(
    [
        joint_df['in_ssusa'] & joint_df['in_iucn'],
        joint_df['in_ssusa'] & ~joint_df['in_iucn'],
        ~joint_df['in_ssusa'] & joint_df['in_iucn'],
    ],
    ['both', 'ssusa_only', 'iucn_only'],
    default='unknown',
)

# Drop boolean helper columns
joint_df = joint_df.drop(columns=['in_ssusa', 'in_iucn'])

# Summary
print(f'Joint table: {len(joint_df):,} rows (Array × Species)')
print(f'Unique arrays: {joint_df["Array"].nunique()}')
print(f'Unique species: {joint_df["species"].nunique()}')
print(f'\nCategory counts:')
print(joint_df['Category'].value_counts().to_string())
joint_df.head(10)

Joint table: 12,706 rows (Array × Species)
Unique arrays: 261
Unique species: 312

Category counts:
Category
iucn_only     9657
both          2747
ssusa_only     302


,Array,species,Category
0,ARNWR,blarina brevicauda,iucn_only
1,ARNWR,blarina carolinensis,iucn_only
2,ARNWR,canis latrans,both
3,ARNWR,canis rufus,both
4,ARNWR,condylura cristata,iucn_only
5,ARNWR,corynorhinus rafinesquii,iucn_only
6,ARNWR,cryptotis parva,iucn_only
7,ARNWR,didelphis virginiana,both
8,ARNWR,eptesicus fuscus,iucn_only
9,ARNWR,glaucomys volans,iucn_only


## 8. Merge PanTHERIA Life-History Traits

In [ ]:
pantheria_path = os.path.join('..', 'data', 'PanTHERIA_1-0_WR05_Aug2008.txt')

pantheria_df = pd.read_csv(pantheria_path, sep='\t', na_values=['-999', '-999.00'])
print(f'PanTHERIA loaded: {len(pantheria_df):,} species, {len(pantheria_df.columns)} columns')
pantheria_df.head(3)

In [22]:
# Standardize binomial for merge
pantheria_df['Sci_Name'] = pantheria_df['MSW05_Binomial'].str.lower().str.strip()

# Select and rename trait columns
trait_cols = [
    'Sci_Name',
    '1-1_ActivityCycle', '5-1_AdultBodyMass_g', '8-1_AdultForearmLen_mm',
    '13-1_AdultHeadBodyLen_mm', '2-1_AgeatEyeOpening_d', '3-1_AgeatFirstBirth_d',
    '18-1_BasalMetRate_mLO2hr', '6-1_DietBreadth', '7-1_DispersalAge_d',
    '9-1_GestationLen_d', '12-1_HabitatBreadth', '22-1_HomeRange_km2',
    '22-2_HomeRange_Indiv_km2', '14-1_InterbirthInterval_d', '15-1_LitterSize',
    '16-1_LittersPerYear', '17-1_MaxLongevity_m', '5-3_NeonateBodyMass_g',
    '21-1_PopulationDensity_n/km2', '10-1_PopulationGrpSize',
    '23-1_SexualMaturityAge_d', '10-2_SocialGrpSize', '24-1_TeatNumber',
    '12-2_Terrestriality', '6-2_TrophicLevel', '25-1_WeaningAge_d',
    '5-4_WeaningBodyMass_g', '26-1_GR_Area_km2',
]

trait_rename = {
    '1-1_ActivityCycle':           'Activity_Cycle',
    '5-1_AdultBodyMass_g':         'Adult_Body_Mass_g',
    '8-1_AdultForearmLen_mm':      'Adult_Forearm_Len_mm',
    '13-1_AdultHeadBodyLen_mm':    'Adult_Head_Body_Len_mm',
    '2-1_AgeatEyeOpening_d':       'Age_Eye_Opening_d',
    '3-1_AgeatFirstBirth_d':       'Age_First_Birth_d',
    '18-1_BasalMetRate_mLO2hr':    'Basal_Met_Rate_mLO2hr',
    '6-1_DietBreadth':             'Diet_Breadth',
    '7-1_DispersalAge_d':          'Dispersal_Age_d',
    '9-1_GestationLen_d':          'Gestation_Len_d',
    '12-1_HabitatBreadth':         'Habitat_Breadth',
    '22-1_HomeRange_km2':          'Home_Range_km2',
    '22-2_HomeRange_Indiv_km2':    'Home_Range_Indiv_km2',
    '14-1_InterbirthInterval_d':   'Interbirth_Interval_d',
    '15-1_LitterSize':             'Litter_Size',
    '16-1_LittersPerYear':         'Litters_Per_Year',
    '17-1_MaxLongevity_m':         'Max_Longevity_months',
    '5-3_NeonateBodyMass_g':       'Neonate_Body_Mass_g',
    '21-1_PopulationDensity_n/km2':'Population_Density_per_km2',
    '10-1_PopulationGrpSize':      'Population_Group_Size',
    '23-1_SexualMaturityAge_d':    'Sexual_Maturity_Age_d',
    '10-2_SocialGrpSize':          'Social_Group_Size',
    '24-1_TeatNumber':             'Teat_Number',
    '12-2_Terrestriality':         'Terrestriality',
    '6-2_TrophicLevel':            'Trophic_Level',
    '25-1_WeaningAge_d':           'Weaning_Age_d',
    '5-4_WeaningBodyMass_g':       'Weaning_Body_Mass_g',
    '26-1_GR_Area_km2':            'Geographic_Range_Area_km2',
}

pantheria_traits = pantheria_df[trait_cols].copy()
pantheria_traits = pantheria_traits.rename(columns=trait_rename)
trait_cols_readable = ['Sci_Name'] + list(trait_rename.values())

print(f'Selected {len(trait_rename)} trait columns (renamed for readability).')

Selected 28 trait columns (renamed for readability).


In [23]:
# Reverse mapping: IUCN name → PanTHERIA (MSW05) name
pantheria_name_map = {
    'neogale frenata': 'mustela frenata',
    'clethrionomys gapperi': 'myodes gapperi',
    'neogale vison': 'neovison vison',
    'martes pennanti': 'pekania pennanti'
}

joint_df['pantheria_key'] = joint_df['species'].replace(pantheria_name_map)

joint_df = pd.merge(
    joint_df,
    pantheria_traits,
    left_on='pantheria_key',
    right_on='Sci_Name',
    how='left',
    suffixes=('', '_pantheria')
)
joint_df = joint_df.drop(columns=['pantheria_key', 'Sci_Name_pantheria'], errors='ignore')

trait_coverage = joint_df[trait_cols_readable[1:]].notna().sum().sort_values(ascending=False)
print(f'Traits merged. Coverage (non-null counts):')
print(trait_coverage.to_string())

Traits merged. Coverage (non-null counts):
Adult_Body_Mass_g             11357
Litter_Size                   11264
Geographic_Range_Area_km2     11202
Diet_Breadth                  10446
Trophic_Level                 10446
Neonate_Body_Mass_g           10186
Habitat_Breadth                9937
Terrestriality                 9937
Weaning_Age_d                  9899
Gestation_Len_d                9860
Max_Longevity_months           9370
Adult_Head_Body_Len_mm         9240
Sexual_Maturity_Age_d          9202
Population_Density_per_km2     8605
Home_Range_km2                 8313
Home_Range_Indiv_km2           8310
Activity_Cycle                 7976
Litters_Per_Year               7927
Basal_Met_Rate_mLO2hr          7470
Age_Eye_Opening_d              7349
Weaning_Body_Mass_g            6387
Teat_Number                    5616
Interbirth_Interval_d          5296
Age_First_Birth_d              3592
Dispersal_Age_d                3421
Social_Group_Size              2932
Adult_Forearm_Len_mm 

## 9. Merge ITIS Common Names

In [ ]:
# Query ITIS for English common names
itis_path = os.path.join('..', 'data', 'ITIS.sqlite')
conn = sqlite3.connect(itis_path)

query = """
SELECT t.complete_name AS itis_sci_name,
       v.vernacular_name AS ITIS_Common_Name
FROM taxonomic_units t
JOIN vernaculars v ON t.tsn = v.tsn
WHERE v.language = 'English'
  AND t.name_usage = 'valid'
"""

itis_df = pd.read_sql_query(query, conn)
conn.close()

itis_df['itis_sci_name'] = itis_df['itis_sci_name'].str.lower().str.strip()
itis_df['ITIS_Common_Name'] = itis_df['ITIS_Common_Name'].str.lower().str.strip()
itis_df = itis_df.drop_duplicates(subset='itis_sci_name', keep='first')

print(f'ITIS common names loaded: {len(itis_df):,} species')

In [25]:
# Reverse mapping for ITIS (same taxonomy gap)
itis_name_map = {
    'neogale frenata': 'mustela frenata',
    'clethrionomys gapperi': 'myodes gapperi',
    'neogale vison': 'neovison vison',
    'martes pennanti': 'pekania pennanti'
}

joint_df['itis_key'] = joint_df['species'].replace(itis_name_map)

joint_df = pd.merge(
    joint_df,
    itis_df,
    left_on='itis_key',
    right_on='itis_sci_name',
    how='left'
)
joint_df = joint_df.drop(columns=['itis_key', 'itis_sci_name'], errors='ignore')

matched = joint_df['ITIS_Common_Name'].notna().sum()
total = len(joint_df)
print(f'ITIS Common Name matched: {matched:,} / {total:,} rows ({matched/total*100:.1f}%)')
print(f'Species with ITIS name: {joint_df[joint_df["ITIS_Common_Name"].notna()]["species"].nunique()} / {joint_df["species"].nunique()}')

missing_itis = joint_df[joint_df['ITIS_Common_Name'].isna()]['species'].unique()
if len(missing_itis) > 0:
    print(f'\nSpecies missing ITIS common name ({len(missing_itis)}):')
    for sp in sorted(missing_itis):
        print(f'  - {sp}')

ITIS Common Name matched: 11,563 / 12,706 rows (91.0%)
Species with ITIS name: 280 / 312

Species missing ITIS common name (32):
  - alexandromys oeconomus
  - bison bison
  - blarina brevicauda
  - clethrionomys californicus
  - clethrionomys rutilus
  - cryptotis parva
  - erethizon dorsatum
  - heteromys irroratus
  - lasiurus cinereus
  - lasiurus ega
  - lasiurus intermedius
  - lasiurus xanthinus
  - myotis melanorhinus
  - neogale frenata
  - neogale vison
  - neotamias amoenus
  - neotamias cinereicollis
  - neotamias dorsalis
  - neotamias merriami
  - neotamias minimus
  - neotamias obscurus
  - neotamias panamintinus
  - neotamias quadrimaculatus
  - neotamias quadrivittatus
  - neotamias ruficaudus
  - neotamias rufus
  - neotamias senex
  - neotamias sonomae
  - neotamias speciosus
  - neotamias townsendii
  - neotamias umbrinus
  - sorex monticola


## 10. Export: Species per Array

In [26]:
# Add SSUSA Common Name to the per-array dataset
ssusa_common_arr = (
    merged_df[['Sci_Name', 'Common_Name']]
    .drop_duplicates(subset='Sci_Name')
    .rename(columns={'Common_Name': 'SSUSA_Common_Name'})
)
final_df = joint_df.merge(ssusa_common_arr, left_on='species', right_on='Sci_Name', how='left')
final_df = final_df.drop(columns=['Sci_Name'], errors='ignore')

# Rename 'species' → 'Sci_Name'
final_df = final_df.rename(columns={'species': 'Sci_Name'})

# Arrange columns: Array, Sci_Name, SSUSA_Common_Name, ITIS_Common_Name, Category, then traits
trait_col_names_arr = list(trait_rename.values())
ordered_arr_cols = ['Array', 'Sci_Name', 'SSUSA_Common_Name', 'ITIS_Common_Name', 'Category'] + trait_col_names_arr
final_df = final_df[ordered_arr_cols].sort_values(['Array', 'Sci_Name']).reset_index(drop=True)

final_df.head(10)

,Array,Sci_Name,SSUSA_Common_Name,ITIS_Common_Name,Category,Activity_Cycle,Adult_Body_Mass_g,Adult_Forearm_Len_mm,Adult_Head_Body_Len_mm,Age_Eye_Opening_d,Age_First_Birth_d,Basal_Met_Rate_mLO2hr,Diet_Breadth,Dispersal_Age_d,Gestation_Len_d,Habitat_Breadth,Home_Range_km2,Home_Range_Indiv_km2,Interbirth_Interval_d,Litter_Size,Litters_Per_Year,Max_Longevity_months,Neonate_Body_Mass_g,Population_Density_per_km2,Population_Group_Size,Sexual_Maturity_Age_d,Social_Group_Size,Teat_Number,Terrestriality,Trophic_Level,Weaning_Age_d,Weaning_Body_Mass_g,Geographic_Range_Area_km2
0,ARNWR,blarina brevicauda,northern short-tailed shrew,NaN,iucn_only,1.0,18.56,NaN,120.82,18.50,64.0,59.36,6.0,NaN,20.58,2.0,0.003540,0.004261,NaN,5.39,3.0,33.0,0.89,922.93,1.0,71.19,NaN,6.0,1.0,2.0,21.91,NaN,4102748.94
1,ARNWR,blarina carolinensis,NaN,southern short-tailed shrew,iucn_only,NaN,11.91,NaN,73.20,NaN,NaN,NaN,2.0,NaN,NaN,2.0,0.009958,0.009556,NaN,3.95,NaN,NaN,NaN,629.99,NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,868525.35
2,ARNWR,canis latrans,coyote,coyote,both,2.0,11989.10,NaN,872.39,11.94,365.0,3699.00,1.0,255.0,61.74,1.0,18.880000,19.910000,365.00,5.72,NaN,262.0,200.01,0.25,NaN,372.90,NaN,8.0,1.0,3.0,43.71,NaN,17099094.30
3,ARNWR,canis rufus,red wolf,red wolf,both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARNWR,condylura cristata,NaN,star-nosed mole,iucn_only,2.0,48.15,NaN,115.65,NaN,NaN,NaN,2.0,NaN,40.00,3.0,0.003656,0.003673,NaN,5.39,1.0,36.0,1.50,4999.99,1.5,304.16,NaN,8.0,1.0,3.0,20.84,NaN,3692908.52
5,ARNWR,corynorhinus rafinesquii,NaN,eastern big-eared bat,iucn_only,NaN,9.15,43.00,NaN,9.00,NaN,NaN,1.0,NaN,55.27,1.0,NaN,NaN,NaN,1.00,1.0,121.2,2.45,NaN,144.0,210.45,NaN,NaN,2.0,3.0,21.00,NaN,1333256.65
6,ARNWR,cryptotis parva,NaN,NaN,iucn_only,2.0,4.97,NaN,67.99,14.00,NaN,44.66,6.0,NaN,21.99,1.0,0.001442,0.001725,NaN,4.55,2.5,21.0,0.35,493.99,15.0,35.44,NaN,6.0,1.0,2.0,20.34,NaN,4102791.80
7,ARNWR,didelphis virginiana,virginia opossum,virginia opossum,both,1.0,2442.08,NaN,NaN,NaN,186.0,719.07,4.0,107.5,12.69,2.0,0.580000,0.510000,136.87,8.62,2.0,60.0,0.15,18.08,NaN,225.55,1.0,13.0,2.0,2.0,109.62,125.0,5696037.14
8,ARNWR,eptesicus fuscus,NaN,big brown bat,iucn_only,NaN,17.49,46.69,NaN,1.50,NaN,30.42,1.0,NaN,52.14,1.0,NaN,NaN,NaN,1.85,1.0,240.0,3.21,NaN,50.0,395.29,NaN,NaN,2.0,3.0,34.18,13.2,13321726.93
9,ARNWR,glaucomys volans,mexican flying squirrel,southern flying squirrel,iucn_only,1.0,71.90,NaN,127.98,26.00,NaN,67.05,5.0,NaN,39.49,1.0,0.005346,0.008786,116.75,3.13,2.0,144.0,3.61,447.59,NaN,335.63,NaN,NaN,2.0,2.0,57.23,46.2,3850610.98


In [27]:
# Quick summary
print(f'Final shape: {final_df.shape}')
print(f'Unique arrays: {final_df["Array"].nunique()}')
print(f'Unique species: {final_df["Sci_Name"].nunique()}')
print(f'\nCategory breakdown:')
print(final_df['Category'].value_counts().to_string())
print(f'\nColumns:')
for i, col in enumerate(final_df.columns, 1):
    print(f'  {i:2d}. {col}')

Final shape: (12706, 33)
Unique arrays: 261
Unique species: 312

Category breakdown:
Category
iucn_only     9657
both          2747
ssusa_only     302

Columns:
   1. Array
   2. Sci_Name
   3. SSUSA_Common_Name
   4. ITIS_Common_Name
   5. Category
   6. Activity_Cycle
   7. Adult_Body_Mass_g
   8. Adult_Forearm_Len_mm
   9. Adult_Head_Body_Len_mm
  10. Age_Eye_Opening_d
  11. Age_First_Birth_d
  12. Basal_Met_Rate_mLO2hr
  13. Diet_Breadth
  14. Dispersal_Age_d
  15. Gestation_Len_d
  16. Habitat_Breadth
  17. Home_Range_km2
  18. Home_Range_Indiv_km2
  19. Interbirth_Interval_d
  20. Litter_Size
  21. Litters_Per_Year
  22. Max_Longevity_months
  23. Neonate_Body_Mass_g
  24. Population_Density_per_km2
  25. Population_Group_Size
  26. Sexual_Maturity_Age_d
  27. Social_Group_Size
  28. Teat_Number
  29. Terrestriality
  30. Trophic_Level
  31. Weaning_Age_d
  32. Weaning_Body_Mass_g
  33. Geographic_Range_Area_km2


In [28]:
output_path = os.path.join(output_dir, 'species_per_array_iucn_traits.csv')
final_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to data/traits/species_per_array_iucn_traits.csv


## 11. Export: Unique Species List

In [29]:
# ---------- Determine overall Category per species (across all arrays) ----------
species_cats = joint_df[['species', 'Category']].drop_duplicates()
species_in_ssusa = species_cats[species_cats['Category'].isin(['both', 'ssusa_only'])]['species'].unique()
species_in_iucn  = species_cats[species_cats['Category'].isin(['both', 'iucn_only'])]['species'].unique()

all_species = joint_df['species'].unique()
cat_rows = []
for sp in all_species:
    in_s = sp in species_in_ssusa
    in_i = sp in species_in_iucn
    if in_s and in_i:
        cat_rows.append((sp, 'both'))
    elif in_s:
        cat_rows.append((sp, 'ssusa_only'))
    else:
        cat_rows.append((sp, 'iucn_only'))

species_category = pd.DataFrame(cat_rows, columns=['species', 'Category'])

# ---------- SSUSA Common Name (one per species) ----------
ssusa_common = (
    merged_df[['Sci_Name', 'Common_Name']]
    .drop_duplicates(subset='Sci_Name')
    .rename(columns={'Sci_Name': 'species', 'Common_Name': 'SSUSA_Common_Name'})
)

# ---------- Trait columns (one row per species — take first non-null) ----------
trait_col_names = list(trait_rename.values())
species_traits = (
    joint_df[['species'] + trait_col_names]
    .drop_duplicates(subset='species')
    .reset_index(drop=True)
)

# ---------- ITIS Common Name (one per species) ----------
species_itis = (
    joint_df[['species', 'ITIS_Common_Name']]
    .drop_duplicates(subset='species')
    .reset_index(drop=True)
)

# ---------- Assemble unique species table ----------
unique_species_df = species_category.merge(ssusa_common, on='species', how='left') \
                                     .merge(species_itis, on='species', how='left') \
                                     .merge(species_traits, on='species', how='left')

# Rename species → Sci_Name
unique_species_df = unique_species_df.rename(columns={'species': 'Sci_Name'})

# ---------- Arrange columns: Sci_Name, SSUSA_Common_Name, ITIS_Common_Name, Category, then traits ----------
ordered_cols = ['Sci_Name', 'SSUSA_Common_Name', 'ITIS_Common_Name', 'Category'] + trait_col_names
unique_species_df = unique_species_df[ordered_cols].sort_values('Sci_Name').reset_index(drop=True)

print(f'Unique species list: {len(unique_species_df)} species')
print(f'\nCategory breakdown:')
print(unique_species_df['Category'].value_counts().to_string())
print(f'\nColumns ({len(unique_species_df.columns)}):')
for i, c in enumerate(unique_species_df.columns, 1):
    print(f'  {i:2d}. {c}')
unique_species_df.head(10)

Unique species list: 312 species

Category breakdown:
Category
iucn_only     199
both          105
ssusa_only      8

Columns (32):
   1. Sci_Name
   2. SSUSA_Common_Name
   3. ITIS_Common_Name
   4. Category
   5. Activity_Cycle
   6. Adult_Body_Mass_g
   7. Adult_Forearm_Len_mm
   8. Adult_Head_Body_Len_mm
   9. Age_Eye_Opening_d
  10. Age_First_Birth_d
  11. Basal_Met_Rate_mLO2hr
  12. Diet_Breadth
  13. Dispersal_Age_d
  14. Gestation_Len_d
  15. Habitat_Breadth
  16. Home_Range_km2
  17. Home_Range_Indiv_km2
  18. Interbirth_Interval_d
  19. Litter_Size
  20. Litters_Per_Year
  21. Max_Longevity_months
  22. Neonate_Body_Mass_g
  23. Population_Density_per_km2
  24. Population_Group_Size
  25. Sexual_Maturity_Age_d
  26. Social_Group_Size
  27. Teat_Number
  28. Terrestriality
  29. Trophic_Level
  30. Weaning_Age_d
  31. Weaning_Body_Mass_g
  32. Geographic_Range_Area_km2


,Sci_Name,SSUSA_Common_Name,ITIS_Common_Name,Category,Activity_Cycle,Adult_Body_Mass_g,Adult_Forearm_Len_mm,Adult_Head_Body_Len_mm,Age_Eye_Opening_d,Age_First_Birth_d,Basal_Met_Rate_mLO2hr,Diet_Breadth,Dispersal_Age_d,Gestation_Len_d,Habitat_Breadth,Home_Range_km2,Home_Range_Indiv_km2,Interbirth_Interval_d,Litter_Size,Litters_Per_Year,Max_Longevity_months,Neonate_Body_Mass_g,Population_Density_per_km2,Population_Group_Size,Sexual_Maturity_Age_d,Social_Group_Size,Teat_Number,Terrestriality,Trophic_Level,Weaning_Age_d,Weaning_Body_Mass_g,Geographic_Range_Area_km2
0,alces alces,moose,eurasian elk,both,2.0,461900.76,NaN,2930.47,NaN,1216.66,NaN,1.0,NaN,235.00,1.0,71.750000,73.260000,365.0,1.25,1.70,324.0,12999.99,0.40,NaN,668.20,1.0,4.0,1.0,1.0,98.85,86299.96,NaN
1,alexandromys oeconomus,NaN,NaN,iucn_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ammospermophilus harrisii,harris's antelope squirrel,harris' antelope squirrel,both,3.0,126.57,NaN,159.17,31.50,NaN,NaN,4.0,NaN,29.19,NaN,NaN,NaN,NaN,6.32,1.00,NaN,3.59,24.97,NaN,198.57,NaN,NaN,NaN,1.0,48.63,NaN,270261.06
3,ammospermophilus interpres,NaN,texas antelope squirrel,iucn_only,3.0,112.53,NaN,151.34,NaN,NaN,NaN,4.0,NaN,NaN,2.0,NaN,NaN,NaN,9.36,1.25,NaN,NaN,NaN,NaN,NaN,NaN,10.0,1.0,2.0,NaN,NaN,313517.36
4,ammospermophilus leucurus,white-tailed antelope squirrel,white-tailed antelope squirrel,both,3.0,103.92,NaN,151.14,35.00,NaN,93.88,5.0,NaN,29.38,2.0,0.040000,0.050000,NaN,8.00,1.25,69.6,2.90,14.49,NaN,377.37,NaN,NaN,1.0,2.0,64.51,34.99,990897.69
5,ammospermophilus nelsoni,NaN,nelson's antelope squirrel,iucn_only,3.0,160.42,NaN,168.38,NaN,NaN,NaN,5.0,NaN,26.00,2.0,0.030000,0.040000,NaN,8.90,1.00,66.0,4.87,NaN,NaN,377.57,7.0,14.0,1.0,2.0,29.83,41.00,4125.09
6,antilocapra americana,pronghorn,pronghorn,both,NaN,47450.01,NaN,1310.00,NaN,365.00,8960.00,2.0,NaN,247.99,1.0,8.050000,9.530000,365.0,1.93,1.00,144.0,3454.70,0.82,1000.0,534.68,12.0,NaN,1.0,1.0,87.54,9662.00,2009745.94
7,antrozous pallidus,NaN,pallid bat,iucn_only,NaN,22.24,54.53,NaN,6.25,NaN,18.70,2.0,NaN,59.01,1.0,NaN,NaN,NaN,1.78,NaN,109.2,3.09,NaN,60.0,NaN,NaN,NaN,2.0,3.0,61.10,13.16,4228781.56
8,aplodontia rufa,mountain beaver,mountain beaver,both,NaN,806.21,NaN,306.95,NaN,730.00,414.73,2.0,NaN,30.00,1.0,0.001352,0.001686,NaN,2.46,1.00,120.0,23.12,778.39,NaN,827.69,NaN,NaN,1.0,1.0,58.40,347.00,245595.72
9,arborimus albipes,NaN,white-footed vole,iucn_only,NaN,23.00,NaN,103.72,NaN,NaN,NaN,3.0,NaN,NaN,2.0,NaN,NaN,NaN,2.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,1.0,NaN,NaN,77783.68


In [30]:
# Save unique species list
unique_species_path = os.path.join(output_dir, 'unique_species_traits.csv')
unique_species_df.to_csv(unique_species_path, index=False)
print(f'Saved to {unique_species_path}')

Saved to data/traits/unique_species_traits.csv


## 12. Diagnostics: Missing Traits & Common Names

In [31]:
# ---------- Species missing PanTHERIA traits ----------
# A species is "missing traits" if ALL 28 trait columns are NaN
trait_col_list = list(trait_rename.values())
has_any_trait = unique_species_df[trait_col_list].notna().any(axis=1)
missing_traits_df = unique_species_df[~has_any_trait][['Sci_Name', 'SSUSA_Common_Name', 'ITIS_Common_Name', 'Category']].copy()

print(f'=== Species with NO PanTHERIA traits ({len(missing_traits_df)}) ===')
if len(missing_traits_df) > 0:
    for _, row in missing_traits_df.iterrows():
        print(f"  {row['Sci_Name']:40s}  Category: {row['Category']:12s}  SSUSA: {str(row['SSUSA_Common_Name']):30s}  ITIS: {str(row['ITIS_Common_Name'])}")
else:
    print('  (none — all species have at least one trait)')

print()

# ---------- Species missing ITIS Common Name ----------
missing_itis_df = unique_species_df[unique_species_df['ITIS_Common_Name'].isna()][['Sci_Name', 'SSUSA_Common_Name', 'Category']].copy()

print(f'=== Species with NO ITIS Common Name ({len(missing_itis_df)}) ===')
if len(missing_itis_df) > 0:
    for _, row in missing_itis_df.iterrows():
        print(f"  {row['Sci_Name']:40s}  Category: {row['Category']:12s}  SSUSA: {str(row['SSUSA_Common_Name'])}")
else:
    print('  (none — all species have an ITIS common name)')

print()

# ---------- Species missing BOTH ----------
missing_both_df = missing_traits_df[missing_traits_df['ITIS_Common_Name'].isna()]
print(f'=== Species missing BOTH traits AND ITIS name ({len(missing_both_df)}) ===')
if len(missing_both_df) > 0:
    for _, row in missing_both_df.iterrows():
        print(f"  {row['Sci_Name']:40s}  Category: {row['Category']:12s}  SSUSA: {str(row['SSUSA_Common_Name'])}")
else:
    print('  (none)')

=== Species with NO PanTHERIA traits (48) ===
  alexandromys oeconomus                    Category: iucn_only     SSUSA: nan                             ITIS: nan
  callospermophilus lateralis               Category: both          SSUSA: golden mantled ground squirrel  ITIS: golden-mantled ground squirrel
  callospermophilus saturatus               Category: iucn_only     SSUSA: nan                             ITIS: cascade golden-mantled ground squirrel
  canis rufus                               Category: both          SSUSA: red wolf                        ITIS: red wolf
  cervus canadensis                         Category: both          SSUSA: elk                             ITIS: wapiti
  clethrionomys californicus                Category: iucn_only     SSUSA: nan                             ITIS: nan
  clethrionomys rutilus                     Category: iucn_only     SSUSA: nan                             ITIS: nan
  heteromys irroratus                       Category: iucn_only  